# 人工对照实验

阅读 `../docs/human_evaluation.md` 和 `../benchmarks/HUMAN_COMPARISON.md`。16 个配对单元是待完成的实验设计，尚无有效时间或准确性提升结论。按 human_first / agent_first 分组执行并记录真实主动时间。


In [ ]:
from pathlib import Path
import json,sys,os,subprocess
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'benchmarks/prepare.py').is_file() and (p/'evaluator/runner.py').is_file()),None)
if ROOT is None: raise RuntimeError('请打开完整项目根目录')
ENV=dict(os.environ,PYTHONIOENCODING='utf-8',PYTHONUTF8='1')
ORDER='human_first'  # 第二阶段改成agent_first
CAMPAIGN='comparison_v1'  # 独立于expanded_v1
for r in json.loads((ROOT/'benchmarks/human_comparison.json').read_text(encoding='utf-8'))['records']:
    if r['order']==ORDER: print(r['pair_id'],r['question'])


In [ ]:
if 'JOB' in globals() and JOB.poll() is None: raise RuntimeError('上一组仍在运行')
r=subprocess.run([sys.executable,str(ROOT/'benchmarks/prepare.py'),'--project-root',str(ROOT),'--campaign',CAMPAIGN,'--batch',ORDER,'--mode','rules'],env=ENV,capture_output=True,text=True,encoding='utf-8',errors='replace')
if r.returncode: raise RuntimeError(r.stderr)
PLAN=json.loads(r.stdout);print(json.dumps(PLAN,ensure_ascii=False,indent=2))
CONFIG=Path(PLAN['config_path']) if PLAN.get('config_path') else None


In [ ]:
RUN_COMPARISON=False
if RUN_COMPARISON:
    if CONFIG is None: raise RuntimeError('该组已完成')
    if 'JOB' in globals() and JOB.poll() is None: raise RuntimeError('仍有运行任务')
    cfg=json.loads(CONFIG.read_text(encoding='utf-8'))
    if Path(cfg['output_dir']).exists(): raise RuntimeError('请重新准备配置')
    LOG=CONFIG.with_suffix('.log')
    with LOG.open('w',encoding='utf-8') as f:
        JOB=subprocess.Popen([sys.executable,'runner.py','--config',str(CONFIG)],cwd=ROOT/'evaluator',env=ENV,stdout=f,stderr=subprocess.STDOUT)
    print('对照结果目录：',cfg['output_dir'])
else: print('未启动；主动操作时间还需自己计时，不能直接用浏览器运行耗时代替。')


In [ ]:
if 'JOB' in globals():
    print(LOG.read_text(encoding='utf-8',errors='replace')[-16000:])
    print('返回码：',JOB.poll())


完成两边交付后填写human_comparison.json。工具侧启动和汇总的共用主动时间按8项均摊，并在note记录分配方法；不要对每项重复计入整批时间。复核各项的时间按实际分别记录。

In [ ]:
r=subprocess.run([sys.executable,str(ROOT/'benchmarks/compare_metrics.py')],env=ENV,capture_output=True,text=True,encoding='utf-8',errors='replace')
print(r.stdout,r.stderr)
if r.returncode: raise RuntimeError('请检查指标填写')
